# parcel_readings.csv

In [25]:
from pathlib import Path

import polars as pl

ROOT = Path(".")
READINGS_PATH = ROOT / "resources" / "parcel_readings.csv"
META_PATH = ROOT / "resources" / "parcel_metadata.csv"

NDVI_MIN = -1.0
NDVI_MAX = 1.0

READINGS_COLS = [
    "parcel_id",
    "date",
    "ndvi_value",
    "temperature_c",
    "rainfall_mm",
    "sensor_status",
]
META_COLS = [
    "parcel_id",
    "mill_id",
    "crop_type",
    "sowing_date",
    "area_hectares",
]

In [26]:
readings = pl.read_csv(
    READINGS_PATH,
    schema_overrides={col: pl.Utf8 for col in READINGS_COLS},
)
N_READINGS = readings.height

print(f"{N_READINGS:,} rows x {readings.width} cols")
display(readings.head(5))

3,447 rows x 6 cols


parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status
str,str,str,str,str,str
"""PARCEL_018""","""16/05/2026""","""0.595""","""15.4""","""0.0""","""error"""
"""PARCEL_014""","""2026-01-27""","""0.457""","""27.6""","""0.0""","""OK"""
"""PARCEL_006""","""2026-05-17""","""0.497""","""25.8""","""0.0""","""OK"""
"""PARCEL_004""","""10/02/2026""","""0.81""","""25.0""","""0.0""","""OK"""
"""PARCEL_002""","""2026-01-17""","""0.269""","""33.2""","""0.0""","""OK"""


In [27]:
print(readings.schema)
display(readings.select("date").head(10))

Schema([('parcel_id', String), ('date', String), ('ndvi_value', String), ('temperature_c', String), ('rainfall_mm', String), ('sensor_status', String)])


date
str
"""16/05/2026"""
"""2026-01-27"""
"""2026-05-17"""
"""10/02/2026"""
"""2026-01-17"""
"""2026-03-19"""
"""2026-05-08"""
"""20-Jan-2026"""
"""2026-05-09"""


### date

In [28]:
def pct(n: int, total: int) -> str:
    if total == 0:
        return "0.0%"
    return f"{100 * n / total:.1f}%"


def date_format_bucket(date_col: str) -> pl.Expr:
    c = pl.col(date_col)
    return (
        pl.when(c.str.contains(r"^\d{4}-\d{2}-\d{2}$"))
        .then(pl.lit("ISO (YYYY-MM-DD)"))
        .when(c.str.contains(r"^\d{2}/\d{2}/\d{4}$"))
        .then(pl.lit("DD/MM/YYYY"))
        .when(c.str.contains(r"^\d{2}-[A-Za-z]{3}-\d{4}$"))
        .then(pl.lit("DD-Mon-YYYY"))
        .otherwise(pl.lit("other"))
        .alias("date_format")
    )


def parse_dates_multi(date_col: str) -> pl.Expr:
    c = pl.col(date_col)
    return (
        pl.coalesce(
            c.str.to_date("%Y-%m-%d", strict=False),
            c.str.to_date("%d/%m/%Y", strict=False),
            c.str.to_date("%d-%b-%Y", strict=False),
        )
        .alias("parsed_date")
    )


display(
    readings.with_columns(date_format_bucket("date"))
    .group_by("date_format")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

date_format,count
str,u32
"""ISO (YYYY-MM-DD)""",2431
"""DD/MM/YYYY""",686
"""DD-Mon-YYYY""",330


In [29]:
parsed = readings.with_columns(parse_dates_multi("date"))
unparsed_dates = parsed.filter(pl.col("parsed_date").is_null()).height
print(f"Unparseable dates: {unparsed_dates} ({pct(unparsed_dates, N_READINGS)})")

ambiguous_slash = (
    readings.filter(pl.col("date").str.contains(r"^\d{2}/\d{2}/\d{4}$"))
    .with_columns(
        pl.col("date").str.split("/").list.get(0).cast(pl.Int64).alias("day"),
        pl.col("date").str.split("/").list.get(1).cast(pl.Int64).alias("month"),
    )
    .filter((pl.col("day") <= 12) & (pl.col("month") <= 12))
)
ambig_date_count = ambiguous_slash.select("date").unique().height
ambig_row_count = ambiguous_slash.height
print(
    f"Ambiguous slash dates (day<=12 and month<=12): "
    f"{ambig_date_count} unique dates, {ambig_row_count} rows "
    f"({pct(ambig_row_count, N_READINGS)})"
)

dup_keys = (
    readings.group_by("parcel_id", "date")
    .agg(pl.len().alias("n"))
    .filter(pl.col("n") > 1)
    .sort("n", descending=True)
)
dup_extra_rows = dup_keys.select((pl.col("n") - 1).alias("extra")).to_series().sum() or 0
print(
    f"Duplicate (parcel_id, date) keys: {dup_keys.height} keys, "
    f"{dup_extra_rows} extra rows ({pct(int(dup_extra_rows), N_READINGS)})"
)
display(dup_keys)

Unparseable dates: 0 (0.0%)
Ambiguous slash dates (day<=12 and month<=12): 60 unique dates, 279 rows (8.1%)
Duplicate (parcel_id, date) keys: 8 keys, 8 extra rows (0.2%)


parcel_id,date,n
str,str,u32
"""PARCEL_024""","""2026-03-01""",2
"""PARCEL_016""","""16/05/2026""",2
"""PARCEL_007""","""2026-04-16""",2
"""PARCEL_011""","""2026-05-05""",2
"""PARCEL_009""","""2026-01-01""",2
"""PARCEL_007""","""2026-05-30""",2
"""PARCEL_008""","""2026-03-27""",2
"""PARCEL_009""","""2026-05-29""",2


### sensor_status

In [30]:
def has_leading_trailing_ws(df: pl.DataFrame, col: str) -> pl.DataFrame:
    return df.filter(pl.col(col) != pl.col(col).str.strip_chars())


def empty_or_null_mask(col: str) -> pl.Expr:
    stripped = pl.col(col).str.strip_chars()
    return stripped.is_null() | (stripped == "") | stripped.str.to_uppercase().is_in(
        ["NA", "NAN", "NULL", "NONE"]
    )


def normalize_sensor_status(col: str = "sensor_status") -> pl.Expr:
    stripped = pl.col(col).str.strip_chars().str.to_uppercase()
    return (
        pl.when(stripped.is_in(["", "NA", "NAN", "NULL", "NONE"]))
        .then(pl.lit("MISSING"))
        .when(stripped == "OK")
        .then(pl.lit("OK"))
        .when(stripped == "ERROR")
        .then(pl.lit("ERROR"))
        .otherwise(stripped)
        .alias("sensor_status_normalized")
    )


print("Raw value counts:")
display(
    readings.group_by("sensor_status")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

ws_status = has_leading_trailing_ws(readings, "sensor_status").height
print(f"Rows with leading/trailing whitespace: {ws_status} ({pct(ws_status, N_READINGS)})")

print("\nAfter strip + uppercase (NA/empty -> MISSING):")
display(
    readings.with_columns(normalize_sensor_status())
    .group_by("sensor_status_normalized")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

missing_status = (
    readings.with_columns(normalize_sensor_status())
    .filter(pl.col("sensor_status_normalized") == "MISSING")
    .height
)
print(f"Missing/NA/empty sensor_status: {missing_status} ({pct(missing_status, N_READINGS)})")

Raw value counts:


sensor_status,count
str,u32
"""OK""",2890
"""Error""",90
"""ERROR""",80
"""error""",76
"""ok""",64
"""OK """,57
""" OK""",53
"""NA""",53
null,43


Rows with leading/trailing whitespace: 110 (3.2%)

After strip + uppercase (NA/empty -> MISSING):


sensor_status_normalized,count
str,u32
"""OK""",3064
"""ERROR""",246
"""MISSING""",94
null,43


Missing/NA/empty sensor_status: 94 (2.7%)


### ndvi_value

In [31]:
ndvi_f = readings.with_columns(
    pl.col("ndvi_value").cast(pl.Float64, strict=False).alias("ndvi_f")
)
ndvi_cast_fail = ndvi_f.filter(pl.col("ndvi_f").is_null() & ~empty_or_null_mask("ndvi_value")).height
ndvi_below = ndvi_f.filter(pl.col("ndvi_f") < NDVI_MIN).height
ndvi_above = ndvi_f.filter(pl.col("ndvi_f") > NDVI_MAX).height
ndvi_out_of_range = ndvi_below + ndvi_above

print(f"NDVI cast failures: {ndvi_cast_fail} ({pct(ndvi_cast_fail, N_READINGS)})")
print(f"NDVI below {NDVI_MIN}: {ndvi_below} ({pct(ndvi_below, N_READINGS)})")
print(f"NDVI above {NDVI_MAX}: {ndvi_above} ({pct(ndvi_above, N_READINGS)})")
print(f"NDVI out of range total: {ndvi_out_of_range} ({pct(ndvi_out_of_range, N_READINGS)})")

if ndvi_out_of_range:
    display(
        ndvi_f.filter((pl.col("ndvi_f") < NDVI_MIN) | (pl.col("ndvi_f") > NDVI_MAX)).select(
            pl.col("ndvi_f").min().alias("min_ndvi"),
            pl.col("ndvi_f").max().alias("max_ndvi"),
        )
    )

NDVI cast failures: 0 (0.0%)
NDVI below -1.0: 51 (1.5%)
NDVI above 1.0: 53 (1.5%)
NDVI out of range total: 104 (3.0%)


min_ndvi,max_ndvi
f64,f64
-1.976,1.997


### temperature_c / rainfall_mm

In [32]:
for col in ["temperature_c", "rainfall_mm"]:
    num = readings.with_columns(pl.col(col).cast(pl.Float64, strict=False).alias("val"))
    cast_fail = num.filter(pl.col("val").is_null() & ~empty_or_null_mask(col)).height
    whole = num.filter(pl.col("val").is_not_null() & (pl.col("val") % 1 == 0)).height
    fractional = num.filter(pl.col("val").is_not_null() & (pl.col("val") % 1 != 0)).height
    print(f"{col}:")
    print(f"  Cast failures: {cast_fail} ({pct(cast_fail, N_READINGS)})")
    print(f"  Whole-number values: {whole} ({pct(whole, N_READINGS)})")
    print(f"  Fractional values: {fractional} ({pct(fractional, N_READINGS)})")
    print()

temperature_c:
  Cast failures: 0 (0.0%)
  Whole-number values: 349 (10.1%)
  Fractional values: 3098 (89.9%)

rainfall_mm:
  Cast failures: 0 (0.0%)
  Whole-number values: 2833 (82.2%)
  Fractional values: 614 (17.8%)



# parcel_metadata.csv

In [33]:
metadata = pl.read_csv(
    META_PATH,
    schema_overrides={col: pl.Utf8 for col in META_COLS},
)
N_METADATA = metadata.height

print(f"{N_METADATA:,} rows x {metadata.width} cols")
print(metadata.schema)
display(metadata.head(5))

28 rows x 5 cols
Schema([('parcel_id', String), ('mill_id', String), ('crop_type', String), ('sowing_date', String), ('area_hectares', String)])


parcel_id,mill_id,crop_type,sowing_date,area_hectares
str,str,str,str,str
"""PARCEL_001""","""MILL_NORTH""","""sugarcane""","""2026-02-10""","""9.03"""
"""PARCEL_002""","""MILL_SOUTH""","""sugarcane""","""2026-01-16""","""8.97"""
"""PARCEL_003""","""MILL_NORTH""","""soybean""","""2026-02-13""","""5.35"""
"""PARCEL_004""","""MILL_NORTH""","""sugarcane""","""2026-01-02""","""3.18"""
"""PARCEL_005""","""MILL_NORTH""","""soybean""","""2026-02-08""","""2.79"""


In [34]:
sowing_fail = (
    metadata.with_columns(
        pl.col("sowing_date").str.to_date("%Y-%m-%d", strict=False).alias("sowing_parsed")
    )
    .filter(pl.col("sowing_parsed").is_null() & ~empty_or_null_mask("sowing_date"))
    .height
)
print(f"sowing_date ISO parse failures: {sowing_fail} ({pct(sowing_fail, N_METADATA)})")

no_readings = (
    metadata.select("parcel_id")
    .join(readings.select("parcel_id").unique(), on="parcel_id", how="anti")
    .sort("parcel_id")
)
print(
    f"Parcels in metadata but not in readings: {no_readings.height} "
    f"({pct(no_readings.height, N_METADATA)})"
)
display(no_readings)

sowing_date ISO parse failures: 0 (0.0%)
Parcels in metadata but not in readings: 3 (10.7%)


parcel_id
str
"""PARCEL_050"""
"""PARCEL_051"""
"""PARCEL_052"""


### join

In [35]:
joined = readings.join(metadata, on="parcel_id", how="left")
unmatched_readings = joined.filter(pl.col("mill_id").is_null()).height
print(
    f"Reading rows without metadata match: {unmatched_readings} "
    f"({pct(unmatched_readings, N_READINGS)})"
)
display(joined.filter(pl.col("mill_id").is_null()).head(10))

orphan_ids = (
    readings.select("parcel_id")
    .unique()
    .join(metadata.select("parcel_id"), on="parcel_id", how="anti")
    .sort("parcel_id")
)
print(f"Distinct orphan parcel_ids: {orphan_ids.height}")
display(orphan_ids)

Reading rows without metadata match: 40 (1.2%)


parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status,mill_id,crop_type,sowing_date,area_hectares
str,str,str,str,str,str,str,str,str,str
"""PARCEL_099""","""19-Mar-2026""","""0.577""","""15.6""","""0.0""","""OK""",null,null,null,null
"""PARCEL_098""","""2026-01-02""","""0.038""","""19.6""","""4.1""","""ok""",null,null,null,null
"""PARCEL_099""","""2026-03-26""","""0.654""","""32.9""","""1.7""","""OK""",null,null,null,null
"""PARCEL_098""","""27/01/2026""","""0.181""","""18.9""","""6.0""","""OK""",null,null,null,null
"""PARCEL_098""","""2026-04-06""","""0.441""","""21.2""","""4.4""","""OK""",null,null,null,null
"""PARCEL_099""","""2026-02-09""","""0.444""","""22.8""","""3.5""","""OK""",null,null,null,null
"""PARCEL_099""","""01/02/2026""","""0.299""","""20.7""","""0.0""","""OK""",null,null,null,null
"""PARCEL_098""","""2026-03-11""","""0.395""","""27.3""","""0.0""","""OK""",null,null,null,null
"""PARCEL_099""","""2026-04-10""","""0.834""","""31.6""","""0.0""","""OK""",null,null,null,null


Distinct orphan parcel_ids: 2


parcel_id
str
"""PARCEL_098"""
"""PARCEL_099"""


In [36]:
date_range = (
    readings.with_columns(parse_dates_multi("date"))
    .filter(pl.col("parsed_date").is_not_null())
    .select(
        pl.col("parsed_date").min().alias("min_date"),
        pl.col("parsed_date").max().alias("max_date"),
    )
)
display(date_range)

out_of_window = (
    readings.with_columns(parse_dates_multi("date"))
    .filter(
        pl.col("parsed_date").is_not_null()
        & (
            (pl.col("parsed_date") < pl.date(2026, 1, 1))
            | (pl.col("parsed_date") > pl.date(2026, 5, 31))
        )
    )
    .height
)
print(f"Readings outside Jan–May 2026: {out_of_window} ({pct(out_of_window, N_READINGS)})")

min_date,max_date
date,date
2026-01-01,2026-05-31


Readings outside Jan–May 2026: 0 (0.0%)
